# OCR Experiment Template

**Experiment:** [Brief description]  
**Date:** YYYY-MM-DD  
**Author:** [Your Name]  
**Goal:** [What are we trying to learn/test?]

## Hypothesis

[State your hypothesis]

## Methodology

[Describe experimental approach]

## Setup

In [ ]:
# Standard imports
import sys
from pathlib import Path
import time
import asyncio
from typing import List, Dict, Any

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme()
plt.rcParams['figure.figsize'] = (12, 6)

# GPU libraries
import torch

# Environment
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print(f"Project root: {project_root}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## Load Test Data

In [ ]:
# Define test documents
test_docs = [
    project_root / "tests" / "fixtures" / "sample_de.pdf",
    # Add more test documents
]

# Verify files exist
for doc in test_docs:
    assert doc.exists(), f"File not found: {doc}"

print(f"Loaded {len(test_docs)} test documents")

## Initialize OCR Backends

In [ ]:
# Import OCR services
# from app.services.ocr.deepseek import DeepSeekOCR
# from app.services.ocr.got_ocr import GOTOCR
# from app.services.ocr.surya_docling import SuryaOCR

# Initialize backends
backends = {
    # "DeepSeek": DeepSeekOCR(),
    # "GOT-OCR": GOTOCR(),
    # "Surya": SuryaOCR(),
}

print(f"Initialized {len(backends)} OCR backends")

## Run Experiments

In [ ]:
# Results storage
results = []

# Run experiments
for doc_path in test_docs:
    print(f"\nProcessing: {doc_path.name}")
    
    for backend_name, backend in backends.items():
        print(f"  {backend_name}...", end=" ")
        
        # GPU memory before
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            mem_before = torch.cuda.memory_allocated() / 1024**3
        
        # Process document
        start_time = time.time()
        try:
            # result = await backend.process(doc_path)
            result = None  # Placeholder
            processing_time = time.time() - start_time
            
            # GPU memory after
            if torch.cuda.is_available():
                mem_peak = torch.cuda.max_memory_allocated() / 1024**3
                mem_used = mem_peak - mem_before
            else:
                mem_used = 0
            
            # Store result
            results.append({
                "document": doc_path.name,
                "backend": backend_name,
                "processing_time": processing_time,
                "vram_used_gb": mem_used,
                # "text_length": len(result.text) if result else 0,
                # "confidence": result.confidence if result else 0,
                "success": True
            })
            
            print(f"✅ {processing_time:.2f}s, {mem_used:.2f} GB VRAM")
            
        except Exception as e:
            print(f"❌ {str(e)}")
            results.append({
                "document": doc_path.name,
                "backend": backend_name,
                "success": False,
                "error": str(e)
            })
        
        # Cleanup GPU memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n✅ Experiments complete")

## Analyze Results

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)
df_success = df[df['success'] == True]

# Display results
df

In [ ]:
# Summary statistics
summary = df_success.groupby('backend').agg({
    'processing_time': ['mean', 'std', 'min', 'max'],
    'vram_used_gb': ['mean', 'max'],
    # 'confidence': 'mean',
}).round(3)

summary

## Visualizations

In [ ]:
# Processing time comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Processing time
df_success.groupby('backend')['processing_time'].mean().plot.bar(ax=ax1)
ax1.set_title('Average Processing Time by Backend')
ax1.set_ylabel('Time (seconds)')
ax1.set_xlabel('Backend')

# VRAM usage
df_success.groupby('backend')['vram_used_gb'].max().plot.bar(ax=ax2, color='orange')
ax2.set_title('Peak VRAM Usage by Backend')
ax2.set_ylabel('VRAM (GB)')
ax2.set_xlabel('Backend')
ax2.axhline(y=13.6, color='r', linestyle='--', label='85% of 16GB')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Processing time distribution
sns.boxplot(data=df_success, x='backend', y='processing_time')
plt.title('Processing Time Distribution')
plt.ylabel('Time (seconds)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Conclusions

### Key Findings

1. [Finding 1]
2. [Finding 2]
3. [Finding 3]

### Recommendations

1. [Recommendation 1]
2. [Recommendation 2]

### Next Steps

- [ ] [Action item 1]
- [ ] [Action item 2]

## Export Results

In [ ]:
# Save results to CSV
output_dir = project_root / "notebooks" / "results"
output_dir.mkdir(exist_ok=True)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_file = output_dir / f"ocr_experiment_{timestamp}.csv"

df.to_csv(output_file, index=False)
print(f"Results saved to: {output_file}")